# Day 4—Measuring the shape of a sign
*Measuring Manuscripts*

A letter isn't one shape but a range of shapes; every scribe and every century draws it a little differently. Today we turn sign images into numbers and ask which signs are most alike—a teaching-scale version of what Isut's *Guesser* does. We'll use made-up signs so the whole pipeline runs; your real images go in at the end.

## 1. Setup

In [ ]:
!pip install scikit-image scikit-learn --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Some sign images to work with

Three archetypes—a cross, a ring, a diagonal—each with several noisy, slightly shifted copies, since no scribe draws the same sign twice.

In [ ]:
rng = np.random.default_rng(0)
S = 16  # image size

def cross():
    a = np.zeros((S, S)); a[S//2-1:S//2+1, :] = 1; a[:, S//2-1:S//2+1] = 1; return a
def ring():
    yy, xx = np.ogrid[:S, :S]; r = (yy - S/2)**2 + (xx - S/2)**2
    a = np.zeros((S, S)); a[(r < (S/2)**2) & (r > (S/2 - 3)**2)] = 1; return a
def diag():
    a = np.zeros((S, S))
    for k in range(S): a[k, min(S-1, k)] = 1; a[k, max(0, k-1)] = 1
    return a

archetypes = {'cross': cross, 'ring': ring, 'diag': diag}
images, labels = [], []
for name, fn in archetypes.items():
    for _ in range(5):
        img = fn() + 0.15 * rng.standard_normal((S, S))   # add noise
        img = np.clip(img, 0, 1)
        images.append(img); labels.append(name)
images = np.array(images)
print(images.shape[0], 'sign images,', S, 'x', S, 'pixels each')

## 3. Look at them

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(8, 5))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(img, cmap='gray_r'); ax.set_title(lab, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. From image to numbers

To compare shapes, each image becomes a row of numbers—a **feature vector**. The simplest features are the pixels themselves, flattened into one row. Real systems use better features; that's the stub at the end.

In [ ]:
features = images.reshape(len(images), -1)   # 16x16 image -> 256 numbers
print('Feature matrix:', features.shape, '(one row per sign)')

## 5. Which signs are most similar?

Cosine similarity scores every pair from 0 (nothing in common) to 1 (identical). For each sign, which are its nearest neighbors?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(features)

def nearest(idx, k=3):
    order = np.argsort(-sim[idx])
    order = [j for j in order if j != idx][:k]
    return [(labels[j], round(float(sim[idx][j]), 2)) for j in order]

for idx in [0, 5, 10]:
    print(f'sign {idx} ({labels[idx]}) -> nearest:', nearest(idx))

## 6. A similarity heatmap

The whole comparison at once. Bright blocks along the diagonal mean each archetype's copies resemble each other more than the others, which is what we'd expect.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(sim, cmap='viridis')
plt.colorbar(label='cosine similarity')
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.title('Every sign vs every sign'); plt.tight_layout(); plt.show()

## 7. Cluster the signs

Group the signs by shape without giving the computer the labels. If the clusters match the archetypes, the features are working. On real data the clusters might track scribes, periods, or genres instead.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
Z = linkage(features, method='ward')
plt.figure(figsize=(9, 3))
dendrogram(Z, labels=labels, leaf_rotation=90)
plt.title('Clustering signs by shape'); plt.tight_layout(); plt.show()

## 8. (Preview of Day 7) Project to two dimensions

Each sign lives in 256-dimensional pixel space, which we can't see, so we compress it to two dimensions. Day 7 is about this; here's a first look.

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2).fit_transform(features)
colors = {'cross': 'tab:blue', 'ring': 'tab:orange', 'diag': 'tab:green'}
plt.figure(figsize=(6, 5))
for name in archetypes:
    pts = xy[[i for i, l in enumerate(labels) if l == name]]
    plt.scatter(pts[:, 0], pts[:, 1], label=name, c=colors[name])
plt.legend(); plt.title('Signs projected to 2-D (PCA)'); plt.show()

## 9. Normalize before comparing

Raw pixels are sensitive to size and position: the same shape, shifted or scaled, looks different to the numbers. Before comparing, we **center** each sign on its ink and **scale** it to a standard size. Watch what this does to two copies of the same shape that started off shifted.

In [ ]:
def normalize(img):
    """Center a sign on its ink so position doesn't dominate the comparison."""
    ys, xs = np.nonzero(img > 0.3)            # which pixels hold ink
    if len(xs) == 0:
        return img.copy()
    cy, cx = ys.mean(), xs.mean()             # center of mass of the ink
    dy = int(round(S/2 - cy)); dx = int(round(S/2 - cx))   # shift needed to center it
    out = np.zeros_like(img)
    ys2, xs2 = ys + dy, xs + dx
    keep = (ys2 >= 0) & (ys2 < S) & (xs2 >= 0) & (xs2 < S)
    out[ys2[keep], xs2[keep]] = img[ys[keep], xs[keep]]
    return out

# The SAME ring shape, once centered and once pushed into a corner.
base = ring()
shifted = np.zeros((S, S))
shifted[0:11, 0:11] = ring()[3:14, 3:14]      # same ring, off-center

from sklearn.metrics.pairwise import cosine_similarity as cos
raw_sim  = cos(base.reshape(1, -1), shifted.reshape(1, -1))[0, 0]
norm_sim = cos(normalize(base).reshape(1, -1), normalize(shifted).reshape(1, -1))[0, 0]
print(f'same shape, off-center -> raw cosine similarity:  {raw_sim:.2f}')
print(f'same shape, off-center -> after centering:        {norm_sim:.2f}')

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(9, 3))
for a, img, t in zip(ax, [base, shifted, normalize(base), normalize(shifted)],
                     ['original', 'shifted', 'normalized', 'shifted, normalized']):
    a.imshow(img, cmap='gray_r'); a.set_title(t, fontsize=9); a.axis('off')
plt.tight_layout(); plt.show()

Normalizing pulls the off-center copy back on top of the centered one, so the score reflects shape rather than where the sign happened to sit in its crop. Be careful, though: centering away position is safe, but normalizing away rotation or slant would erase a real difference between hands if slant is what you're studying.

## 10. Two ways to measure distance

**Cosine similarity** compares the *direction* of two vectors and ignores their length. **Euclidean distance** is the straight-line gap between them and cares about magnitude—how much ink, not just where. They can rank neighbors differently. Here are both, side by side, for the same signs.

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

sim_cos  = cosine_similarity(features)
dist_euc = euclidean_distances(features)

def neighbors_cos(idx, k=3):
    order = [j for j in np.argsort(-sim_cos[idx]) if j != idx][:k]
    return [(labels[j], round(float(sim_cos[idx][j]), 2)) for j in order]

def neighbors_euc(idx, k=3):
    order = [j for j in np.argsort(dist_euc[idx]) if j != idx][:k]
    return [(labels[j], round(float(dist_euc[idx][j]), 2)) for j in order]

for idx in [0, 5, 10]:
    print(f'sign {idx} ({labels[idx]})')
    print('   cosine    (higher = closer):', neighbors_cos(idx))
    print('   euclidean (lower  = closer):', neighbors_euc(idx))

Both usually agree on the broad grouping here, because the archetypes are well separated. On real signs—where thickness and ink quantity vary a lot—the two metrics part company more often, and the choice matters.

## 11. How robust is this to noise?

Real crops are noisy: stray ink, show-through, binarization mistakes. We test the method by adding more and more noise to each sign and asking a simple question—does its nearest neighbor still share its label? That fraction is **nearest-neighbor accuracy**, and we watch it fall as noise rises.

In [ ]:
def nn_accuracy(feat, labs):
    """Fraction of signs whose nearest neighbor shares its label."""
    s = cosine_similarity(feat)
    np.fill_diagonal(s, -1)            # never pick yourself
    nn = np.argmax(s, axis=1)
    return np.mean([labs[i] == labs[nn[i]] for i in range(len(labs))])

noise_levels = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
accs = []
for nl in noise_levels:
    noisy = np.clip(images + nl * rng.standard_normal(images.shape), 0, 1)
    accs.append(nn_accuracy(noisy.reshape(len(noisy), -1), labels))

plt.figure(figsize=(6, 4))
plt.plot(noise_levels, accs, marker='o')
plt.xlabel('added noise'); plt.ylabel('nearest-neighbor accuracy')
plt.ylim(0, 1.05); plt.title('Accuracy degrades as signs get noisier')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for nl, a in zip(noise_levels, accs):
    print(f'noise {nl:.1f}: nearest-neighbor accuracy {a:.2f}')

Accuracy starts near perfect and drops as noise swamps the shape. Where the curve bends is roughly how much damage these raw-pixel features can absorb before they stop telling the archetypes apart. Better features, or normalization, push that bend to the right.

## 12. Which archetypes get confused?

When the method makes a mistake, *which* signs does it mix up? A **confusion** table counts, for each true archetype, what its nearest neighbor was actually labeled. Off-diagonal entries are the confusions—the shapes the features can't yet separate. To make some happen, we run it on a noisier copy of the signs.

In [ ]:
names = list(archetypes.keys())
noisy = np.clip(images + 0.6 * rng.standard_normal(images.shape), 0, 1)
s = cosine_similarity(noisy.reshape(len(noisy), -1))
np.fill_diagonal(s, -1)
nn = np.argmax(s, axis=1)

conf = np.zeros((len(names), len(names)), dtype=int)
for i in range(len(labels)):
    conf[names.index(labels[i]), names.index(labels[nn[i]])] += 1

plt.figure(figsize=(4.5, 4))
plt.imshow(conf, cmap='Blues')
for i in range(len(names)):
    for j in range(len(names)):
        plt.text(j, i, conf[i, j], ha='center', va='center')
plt.xticks(range(len(names)), names); plt.yticks(range(len(names)), names)
plt.xlabel('nearest-neighbor label'); plt.ylabel('true label')
plt.title('Confusion under heavy noise'); plt.tight_layout(); plt.show()

The diagonal is correct matches; everything off it is a confusion. If 'cross' and 'diag' trade places more than either trades with 'ring', that's the features telling you those two shapes share more ink in common than the third. On real signs this is often the most useful output: not just *how well* it works, but *which* letters it can't tell apart.

## 9. The real sign images

> 🔧 **TO BUILD**—replace the made-up shapes with real hieratic sign crops from PaPYrus/Isut. The pipeline doesn't change; only the source of `images` and `labels` does.

In [ ]:
# === TO BUILD: load real sign crops and build `images` + `labels` ===
# from skimage.io import imread
# from skimage.transform import resize
# def load_sign(path):
#     img = imread(path, as_gray=True)
#     return resize(img, (16, 16))
SIGN_IMAGE_FOLDER = None  # a folder of labelled sign crops
# A cleverer feature than raw pixels (stretch): scikit-image's HOG descriptor.
# from skimage.feature import hog

### Talk it over
- Did the clusters in section 7 match the archetypes? Where did a copy land in the wrong group?
- The heatmap and the tree use the same numbers. Which did you find easier to read, and why?
- **Check-in:** is the script itself your variable? What hands, periods, or genres might you compare by shape?